# ARCSIX 5 min and 1 min merge using `sizedistmerge`

This notebook runs the configurable ARCSIX 4-instrument merge workflow with the new package import style. The next cell is the real executable import/setup cell and should be run first.

It keeps the old production settings for both merge cadences:

- 5 min: `MIN_SAMPLES_PER_INST = 50`, `MIN_OVERLAP_S = 120`
- 1 min: `MIN_SAMPLES_PER_INST = 10`, `MIN_OVERLAP_S = 24`
- FIMS lag: `10 s`
- Inlet-flag padding: `10 s`
- Joint fit: `moment="V"`, `space="linear"`, `pair_w=1.0`
- UHSAS RI bounds: `1.3` to `1.8`
- APS density bounds: `950` to `2000 kg m-3`
- FIMS range: `10` to `500 nm`
- Fixed output grid: `100` log bins from `10` to `5000 nm`
- Consensus: `alpha_uhsas=1.5`, `alpha_pops=0.2`, `alpha_aps=2.0`, `lambda=1e-5`, `c=0.5`, `data_space="log10"`

The package runner handles loading, inlet filtering, overlap QC, optional alignment, optimization, plotting, and per-day NetCDF writing. LUTs default to the packaged LUT resources when `LUT_DIR = None`.


In [ ]:
# Executable import/setup cell. Run this cell first.
from __future__ import annotations

from pathlib import Path
import sys

import numpy as np
import pandas as pd

if "/envs/Research/" not in sys.executable:
    raise RuntimeError(
        "Run this notebook with the Research conda kernel. "
        f"Current Python is: {sys.executable}"
    )

import sizedistmerge as sdm
from sizedistmerge import arcsix_merge_production as mp

print("Python:", sys.executable)
print("sizedistmerge:", sdm.__file__)
print("Packaged UHSAS LUT:", sdm.lut_path("uhsas"))
print("Packaged POPS LUT:", sdm.lut_path("pops"))


## Shared settings

`OUTPUT_SUFFIX` is available if you want to distinguish test runs. The default output directory names are cadence-based, not version-based.


In [ ]:
DATES = [
    "2024-05-28", "2024-05-30", "2024-05-31", "2024-06-03",
    "2024-06-05", "2024-06-06", "2024-06-07", "2024-06-10",
    "2024-06-11", "2024-06-13", "2024-07-25", "2024-07-29",
    "2024-07-30", "2024-08-01", "2024-08-02", "2024-08-07",
    "2024-08-08", "2024-08-09", "2024-08-15",
]

DATA_DIR = Path("/Volumes/Hailstone Data/Research Data/ARCSIX_P3B")
OUTPUT_ROOT = Path("/Users/C832577250/Output")
OUTPUT_SUFFIX = ""

LUT_DIR = None  # None means use packaged LUTs via sdm.lut_path(...)
INSTRUMENTS = ("FIMS", "UHSAS", "POPS", "APS")
APPLY_ALIGNMENT = True

FIMS_LAG = 10
INCLOUD_PAD_S = 10
OVERLAP_FREQ = "1s"

MOMENT = "V"
SPACE = "linear"
PAIR_W = 1.0

BOUNDS_UHSAS = ((1.3, 1.8),)
BOUNDS_APS = ((950.0, 2000.0),)

UHSAS_XMIN = None
UHSAS_XMAX = None
FIMS_XMIN = 10
FIMS_XMAX = 500
POPS_XMIN = None
POPS_XMAX = None

FINE_BIN = 100
GLOBAL_XMIN_NM = 10.0
GLOBAL_XMAX_NM = 5000.0
GLOBAL_EDGES = np.logspace(
    np.log10(GLOBAL_XMIN_NM),
    np.log10(GLOBAL_XMAX_NM),
    FINE_BIN + 1,
).astype(float)

ALPHA_UHSAS = 1.5
ALPHA_POPS = 0.2
ALPHA_APS = 2.0
LAMBDA_TIK = 1e-5
C_PUNISH = 0.5
COMBINATION_SPACE = "log10"

FIVE_MIN = {
    "name": "5min",
    "merge_minutes": 5,
    "output_dir": OUTPUT_ROOT / f"arcsix_sizedist_merge_5min{OUTPUT_SUFFIX}",
    "min_samples_per_inst": 50,
    "min_overlap_s": 120,
    "temporal_enabled": False,
}

ONE_MIN = {
    "name": "1min",
    "merge_minutes": 1,
    "output_dir": OUTPUT_ROOT / f"arcsix_sizedist_merge_1min{OUTPUT_SUFFIX}",
    "min_samples_per_inst": 50 / 5,
    "min_overlap_s": 60 * 2 / 5,
    "temporal_enabled": False,
}

RI_IMPROVE_TEMPORAL = {
    "temporal_prior_params": list(mp.DEFAULT_TEMPORAL_PRIOR_PARAMS),
    "temporal_w_uh": 10.0,
    "temporal_w_po": 10.0,
    "temporal_w_rho": 1e-7,
    "smooth_rho": True,
}

POST_MERGE_QC = {
    "high_cost_thresh": 0.2,
    "cutoff_nm": 10.0,
    "k_sigma_warn": 10.0,
    "k_sigma_drop": 20.0,
    "min_points_for_robust": 50,
    "cpc_column": "CNgt10nm",
    "cpc_statistic": "median",
    "plot_lo": 0.0,
    "plot_hi": 8000.0,
}

ICARTT_SHARED = {
    "mission_name": "ARCSIX 2024",
    "pi_name": "Perkins, Russell",
    "pi_affiliation": "Colorado State University",
}


## Period generation

The package runner accepts explicit periods. These helpers reproduce the old notebook's daily chunk boundaries by loading the day, shifting FIMS earlier by `FIMS_LAG`, then applying `mp.split_frames(..., MERGE_PER * 60)`. The package runner then re-loads the periods and applies inlet filtering, overlap QC, min-count QC, optimization, plots, and NetCDF output.


In [ ]:
def build_periods_for_merge(config: dict) -> list[tuple[pd.Timestamp, pd.Timestamp]]:
    periods = []
    seconds = int(round(float(config["merge_minutes"]) * 60))

    for date in DATES:
        frames = mp.load_arcsix_merge_frames_for_day(
            DATA_DIR,
            date,
            instruments=INSTRUMENTS,
            fims_lag=FIMS_LAG,
        )
        day_periods = mp.periods_from_frames(frames, seconds)
        periods.extend(day_periods)
        print(f"{config['name']} {date}: {len(day_periods)} candidate periods")

    print(f"{config['name']} total candidate periods: {len(periods)}")
    return periods


def temporal_kwargs(config: dict) -> dict:
    if not config.get("temporal_enabled", False):
        return {
            "temporal_prior_params": None,
            "temporal_w_uh": 0.0,
            "temporal_w_po": 0.0,
            "temporal_w_rho": 0.0,
            "smooth_rho": True,
        }
    return dict(RI_IMPROVE_TEMPORAL)


def run_package_merge(config: dict):
    periods = build_periods_for_merge(config)
    if not periods:
        raise RuntimeError(f"No candidate periods built for {config['name']}")

    config['output_dir'].mkdir(parents=True, exist_ok=True)
    print(f"Writing {config['name']} merge to: {config['output_dir']}")

    return mp.run_arcsix_merge_for_periods(
        time_periods=periods,
        data_dir=DATA_DIR,
        output_dir=config['output_dir'],
        fims_lag=FIMS_LAG,
        incloud_pad_s=INCLOUD_PAD_S,
        min_samples_per_inst=config["min_samples_per_inst"],
        min_overlap_s=config["min_overlap_s"],
        overlap_freq=OVERLAP_FREQ,
        moment=MOMENT,
        space=SPACE,
        pair_w=PAIR_W,
        bounds_uhsas=BOUNDS_UHSAS,
        bounds_aps=BOUNDS_APS,
        uhsas_xmin=UHSAS_XMIN,
        uhsas_xmax=UHSAS_XMAX,
        fims_xmin=FIMS_XMIN,
        fims_xmax=FIMS_XMAX,
        pops_xmin=POPS_XMIN,
        pops_xmax=POPS_XMAX,
        lut_dir=LUT_DIR,
        fine_bin=FINE_BIN,
        instruments=INSTRUMENTS,
        apply_alignment=APPLY_ALIGNMENT,
        uhsas_combine_weight=ALPHA_UHSAS,
        pops_combine_weight=ALPHA_POPS,
        aps_combine_weight=ALPHA_APS,
        smoothness_lam=LAMBDA_TIK,
        consensus_c=C_PUNISH,
        consensus_data_space=COMBINATION_SPACE,
        output_edges=GLOBAL_EDGES,
        **temporal_kwargs(config),
    )


def run_product_qc_and_icartt(config: dict):
    output_dir = config['output_dir']
    print(f"Running post-merge QC for {config['name']}: {output_dir}")

    qc_result = mp.run_post_merge_product_qc(
        base_dir=output_dir,
        data_dir=DATA_DIR,
        **POST_MERGE_QC,
    )
    print(
        f"QC {config['name']}: total={qc_result.total_chunks}, "
        f"kept={qc_result.kept_chunks}, "
        f"dropped_extreme={qc_result.dropped_extreme_chunks}"
    )
    print("QC table:", qc_result.qc_table_path)
    print("QC NetCDF:", qc_result.qc_netcdf_dir)

    icartt_dir = output_dir / "icartt_from_qc_flagged_nc"
    product_name = "ARCSIX-MERGED-SIZEDIST-1min" if config['name'] == "1min" else "ARCSIX-MERGED-SIZEDIST"
    revision = "R0" if config['name'] == "1min" else "R1"
    data_source_desc = (
        "FIMS-UHSAS-POPS-APS 1 minute merged aerosol size distribution, NASA P-3B aircraft"
        if config['name'] == "1min"
        else "FIMS-UHSAS-POPS-APS merged aerosol size distribution, NASA P-3B aircraft"
    )
    revision_comment = (
        "Revision 0. New 1-minute merged aerosol number size distribution product from FIMS, UHSAS, POPS, and APS. "
        "Retrieved-parameter columns are experimental/diagnostic only. A temporal-overlap requirement is applied: "
        "at least 24 seconds of same-second 4-instrument overlap within each 1-minute output interval. "
        "For analyses of Dp < 50 nm, we recommend using the native FIMS data."
        if config['name'] == "1min"
        else "Revision 1. Retrieved-parameter columns are experimental/diagnostic only. "
        "For analyses of Dp < 50 nm, we recommend using the native FIMS data. "
        "Revision 1 uses a single uniform bin grid with 100 log10-spaced bins spanning 10-5000 nm, "
        "requires at least 2 minutes of same-second 4-instrument overlap within each 5-minute period, "
        "and uses lambda=1e-5 for Tikhonov regularization."
    )

    ict_paths = mp.convert_qc_netcdf_to_icartt(
        input_dir=qc_result.qc_netcdf_dir,
        out_dir=icartt_dir,
        product_name=product_name,
        revision=revision,
        data_source_desc=data_source_desc,
        revision_comment=revision_comment,
        **ICARTT_SHARED,
    )
    print(f"ICARTT {config['name']}: wrote {len(ict_paths)} files to {icartt_dir}")
    return qc_result, ict_paths


## Run 5 min merge

This writes per-day NetCDF and diagnostic plots under `FIVE_MIN["output_dir"]`.


In [ ]:
run_package_merge(FIVE_MIN)


In [ ]:
qc_5min, ict_5min = run_product_qc_and_icartt(FIVE_MIN)

## Run 1 min merge

This writes per-day NetCDF and diagnostic plots under `ONE_MIN["output_dir"]`.

The standard 1-minute merge settings keep temporal regularization off. To run the RI-improve temporal behavior, set `ONE_MIN["temporal_enabled"] = True` before this cell; the weights are in `RI_IMPROVE_TEMPORAL`.


In [ ]:
run_package_merge(ONE_MIN)


In [ ]:
qc_1min, ict_1min = run_product_qc_and_icartt(ONE_MIN)

## Quick output check

This only lists the NetCDF files written by the two merge runs.


In [ ]:
for config in (FIVE_MIN, ONE_MIN):
    files = sorted(config['output_dir'].glob("*/*_sizedist_merged.nc"))
    qc_files = sorted((config['output_dir'] / "qc_flagged_nc").glob("*_sizedist_merged.nc"))
    ict_files = sorted((config['output_dir'] / "icartt_from_qc_flagged_nc").glob("*.ict"))
    print(f"{config['name']}: {len(files)} raw NetCDF files")
    print(f"{config['name']}: {len(qc_files)} QC NetCDF files")
    print(f"{config['name']}: {len(ict_files)} ICARTT files")
    for path in files[:5]:
        print("  raw", path)
    for path in qc_files[:5]:
        print("  qc ", path)
    for path in ict_files[:5]:
        print("  ict", path)
    print()
